# Multimodal: cobertura de imágenes, embeddings SigLIP y SASRec + SigLIP

Primer paso de la versión multimodal del proyecto (frente M). Usa las imágenes de los productos de la metadata de Amazon 2018:

1. **Cobertura (4 datasets):** qué ítems tienen URL de imagen y cuáles responden con una imagen, con fecha. No necesita GPU.
2. **Descarga (Toys):** baja una imagen por ítem, comprueba que se pueda abrir y guarda un manifiesto con su sha256.
3. **Embeddings (Toys):** SigLIP `google/siglip-so400m-patch14-384` en fp32, de la imagen y del título, en el mismo espacio y normalizados.
4. **SASRec + SigLIP (Toys):** el SASRec del protocolo común con el embedding de cada ítem = embedding de id + proyección del embedding SigLIP, con 3 variantes y la grilla del protocolo, frente al SASRec de solo id en la misma GPU. Métricas totales y por popularidad del ítem objetivo.

Se puede retomar: cada paso guarda su avance y se salta lo hecho. Con `PRUEBA = True` (primera sesión) todo corre con 100 ítems y 1 combinación de 2 épocas, en carpetas `*_prueba`. Corre en Colab (guarda en `MyDrive/IIC3633/checkpoints/multimodal/`) y en local, desde `notebooks/multimodal/` (guarda en `checkpoints/multimodal/`). Las decisiones están en [`README.md`](README.md).

In [ ]:
# Preparación de la sesión: correr esta celda primero en cada sesión nueva (se puede repetir sin problema)
import os
import sys
import subprocess
COMMIT = 'cf7ad70bc59e8bd63e4597a277df7fd160bcc95b'   # commit fijo del repo (lo deja escrito quien coordina al publicar la versión)
URL = 'https://github.com/mannicchiarico/iic3633-proyecto-latentr3'
REPO = '/content/iic3633'
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    if COMMIT == 'PENDIENTE':
        raise RuntimeError('Falta fijar COMMIT: pedirlo a quien coordina.')
    if not os.path.isdir(os.path.join(REPO, '.git')):
        subprocess.run(['git', 'clone', '-q', URL, REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '--detach', 'origin/main' if COMMIT == 'main' else COMMIT],
                   check=True)
    print('Repo en el commit', subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True,
                                              text=True).stdout.strip())
    sys.path.insert(0, os.path.join(REPO, 'colab'))
    import preparar
    RUTAS = preparar.preparar('multimodal')
    RAIZ = REPO
else:
    RAIZ = os.path.abspath(os.path.join('..', '..'))   # local: se corre desde notebooks/multimodal/
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')  # solo la GPU dedicada, antes de importar torch
    for carpeta in [os.path.join(RAIZ, 'notebooks'), os.path.join(RAIZ, 'notebooks', 'sasrec'),
                    os.path.abspath('.'), os.path.join(RAIZ, 'colab')]:
        if carpeta not in sys.path:
            sys.path.insert(0, carpeta)
    RUTAS = None

In [ ]:
# Parámetros
CATEGORIAS = ['CDs_and_Vinyl', 'Musical_Instruments', 'Toys_and_Games', 'Video_Games']   # cobertura
CATEGORIA_EMB = 'Toys_and_Games'   # descarga, embeddings y SASRec + SigLIP
PRUEBA = True                      # primera sesión: 100 ítems por paso y 1 combinación de 2 épocas; después, False
N_PROCESOS = 2 if IN_COLAB else 6  # combinaciones de SASRec en paralelo (Colab gratis tiene 2 CPU)

In [ ]:
import json
import time
import hashlib
import zipfile
import contextlib
import numpy as np
import pandas as pd
import torch
import imagenes as im
import sasrec_baseline as sb
import sasrec_mm as mm
from baseline_comun import verificar_gpu, entorno

base = RUTAS['base'] if IN_COLAB else os.path.join(RAIZ, 'repo_latentr3', 'data')
ITEMS = os.path.join(RAIZ, 'notebooks', 'multimodal', 'items')   # generado por extraer_items.py (versionado)
SALIDA = (os.path.join(RUTAS['iic3633'], 'checkpoints', 'multimodal') if IN_COLAB
          else os.path.join(RAIZ, 'checkpoints', 'multimodal'))
CODIGO_SASREC = RUTAS['codigo_sasrec'] if IN_COLAB else os.path.join(RAIZ, 'SASRec.pytorch')
SUFIJO = '_prueba' if PRUEBA else ''
LIMITE = 100 if PRUEBA else None
COBERTURA = os.path.join(SALIDA, 'cobertura' + SUFIJO)
CARPETA_EMB = os.path.join(SALIDA, CATEGORIA_EMB + SUFIJO)
# Las imágenes se leen del disco local (rápido); en Colab, una copia en zip en Drive evita volver a descargarlas
IMAGENES = ('/content/imagenes_' + CATEGORIA_EMB + SUFIJO if IN_COLAB
            else os.path.join(SALIDA, 'imagenes_' + CATEGORIA_EMB + SUFIJO))
ZIP_IMAGENES = os.path.join(CARPETA_EMB, 'imagenes.zip')
MANIFIESTO_IMAGENES = os.path.join(CARPETA_EMB, 'manifiesto_imagenes.csv')
RUTA_NPZ = os.path.join(CARPETA_EMB, 'siglip.npz')
CARPETA_MM = os.path.join(CARPETA_EMB, 'sasrec_mm')
for carpeta in (COBERTURA, CARPETA_EMB, CARPETA_MM):
    os.makedirs(carpeta, exist_ok=True)
sb.verificar_codigo(CODIGO_SASREC)   # sha256 del model.py del port antes de usarlo

if IN_COLAB:
    from monitor import Monitor
    monitor = Monitor(os.path.join(RUTAS['iic3633'], 'colab', 'multimodal')).iniciar()   # gpu_log.csv cada 60 s
else:
    class _SinMonitor:
        def etapa(self, nombre):
            return contextlib.nullcontext()
    monitor = _SinMonitor()
print('Salida:', SALIDA, '| prueba:', PRUEBA, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no')

## 1. Cobertura de imágenes (4 datasets)

Para cada ítem se elige la primera URL de `imageURLHighRes` y, si no hay, la primera de `imageURL` (el campo queda anotado). Se pide la URL y se guarda el estado HTTP, el tipo de contenido y la fecha: los enlaces se caen con el tiempo, así que la cobertura vale para esa fecha. Un ítem "responde con imagen" si el estado es 200 y el tipo empieza con `image/`.

In [ ]:
resumenes = {}
for categoria in CATEGORIAS:
    items = im.cargar_items(ITEMS, categoria)
    with monitor.etapa(f'cobertura {categoria}'):
        resumenes[categoria] = im.revisar_urls(items, os.path.join(COBERTURA, f'cobertura_{categoria}.csv'),
                                               limite=LIMITE)
with open(os.path.join(COBERTURA, 'resumen.json'), 'w', encoding='utf-8') as f:
    json.dump(resumenes, f, indent=1, ensure_ascii=False)
pd.DataFrame(resumenes).T

## 2. Descarga de las imágenes de Toys

Una imagen por ítem (la misma URL de la cobertura). Se guarda solo si PIL la puede abrir; el manifiesto registra URL, estado, tamaño y sha256. Las imágenes no se versionan.

In [ ]:
items_emb = im.cargar_items(ITEMS, CATEGORIA_EMB)
if IN_COLAB and os.path.isfile(ZIP_IMAGENES) and not os.path.isdir(IMAGENES):   # sesión nueva: se recupera de Drive
    with zipfile.ZipFile(ZIP_IMAGENES) as z:
        z.extractall(IMAGENES)
with monitor.etapa(f'descarga {CATEGORIA_EMB}'):
    descarga = im.descargar(items_emb, IMAGENES, MANIFIESTO_IMAGENES, limite=LIMITE)
if IN_COLAB:   # se reescribe el zip solo si hay imágenes nuevas (cada reemplazo va a la papelera de Drive)
    locales = sorted(f for f in os.listdir(IMAGENES) if f.endswith('.img'))
    en_zip = zipfile.ZipFile(ZIP_IMAGENES).namelist() if os.path.isfile(ZIP_IMAGENES) else []
    if len(locales) != len(en_zip):
        with zipfile.ZipFile(ZIP_IMAGENES + '.tmp', 'w', zipfile.ZIP_STORED) as z:
            for archivo in locales:
                z.write(os.path.join(IMAGENES, archivo), archivo)
        os.replace(ZIP_IMAGENES + '.tmp', ZIP_IMAGENES)
descarga

## 3. Embeddings SigLIP de Toys

SigLIP so400m-patch14-384 en fp32. Imagen: los ítems con imagen descargada (los demás quedan en cero y marcados en `tiene_imagen`). Título: todos los ítems, con el texto rellenado a 64 tokens como se entrenó SigLIP. Todo normalizado. El `.npz` queda en Drive y no se versiona; su sha256, el modelo y la revisión van en `manifiesto_embeddings.json`.

In [ ]:
def sha256(ruta):
    with open(ruta, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


ruta_manifiesto_emb = os.path.join(CARPETA_EMB, 'manifiesto_embeddings.json')
if os.path.isfile(RUTA_NPZ) and os.path.isfile(ruta_manifiesto_emb):
    with open(ruta_manifiesto_emb) as f:
        manifiesto_emb = json.load(f)
    assert sha256(RUTA_NPZ) == manifiesto_emb['sha256'], 'siglip.npz no coincide con su manifiesto'
    print('Embeddings ya calculados; se salta.')
else:
    verificar_gpu('embeddings SigLIP', modelo=None, siglip=im.MODELO_SIGLIP, dtype='float32')   # solo en Colab
    rutas_imagenes = im.imagenes_validas(IMAGENES, MANIFIESTO_IMAGENES)
    with monitor.etapa('embeddings SigLIP'):
        siglip, procesador = im.cargar_siglip(device='cuda' if torch.cuda.is_available() else 'cpu')
        emb = im.embeddings(items_emb, rutas_imagenes, siglip, procesador)
        manifiesto_emb = im.guardar_embeddings(RUTA_NPZ, emb, siglip, {'entorno': entorno(), 'descarga': descarga})
    del siglip
    torch.cuda.empty_cache()
{k: v for k, v in manifiesto_emb.items() if k != 'entorno'}

## 4. SASRec + SigLIP (Toys)

El SASRec del protocolo común (`../sasrec/sasrec_baseline.py`: BCE con 1 negativo en la última posición, Adam, early stopping por NDCG@10 en `valid`), con la tabla de embeddings de ítems reemplazada por **id + W·SigLIP** (`sasrec_mm.EmbItemMM`). Variantes:
- **id + imagen**: los ítems sin imagen suman un vector aprendido.
- **id + imagen (título de respaldo)**: los ítems sin imagen usan el embedding SigLIP de su título.
- **id + título**: control, para separar el aporte de la imagen del de la proyección extra.
- **solo id**: el SASRec de siempre, con la combinación ganadora de BCE (paper) en Toys (lr 1e-4, wd 1e-6; `baseline_sasrec.ipynb`, celda 15), vuelto a entrenar en esta misma GPU.

Cada variante multimodal recorre la grilla del protocolo (lr {1e-2, 1e-3, 1e-4} × wd {1e-4, 1e-5, 1e-6}). Cada combinación terminada queda en un JSON en Drive y no se repite al retomar.

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
LRS, WEIGHT_DECAYS = [1e-2, 1e-3, 1e-4], [1e-4, 1e-5, 1e-6]   # grilla del protocolo común
GANADORA_UNIMODAL = (1e-4, 1e-6)                               # SASRec BCE (paper), Toys
if PRUEBA:
    GRILLA, MAX_EPOCAS = [('id + imagen', 1e-3, 1e-5)], 2
else:
    GRILLA = [(v, lr, wd) for v in mm.VARIANTES for lr in LRS for wd in WEIGHT_DECAYS]
    GRILLA.append((mm.VARIANTE_UNIMODAL, *GANADORA_UNIMODAL))
    MAX_EPOCAS = sb.MAX_EPOCAS


def archivo(variante, lr, wd):
    return os.path.join(CARPETA_MM, f'{mm.NOMBRES[variante]}__lr{lr:g}_wd{wd:g}.json')


pendientes = [g for g in GRILLA if not os.path.isfile(archivo(*g))]
verificar_gpu('SASRec + SigLIP', procesos=N_PROCESOS, combinaciones_pendientes=len(pendientes))   # solo en Colab
inicio = time.time()
with monitor.etapa('SASRec + SigLIP'):
    with ProcessPoolExecutor(N_PROCESOS, mp_context=multiprocessing.get_context('spawn'),
                             initializer=sb.inicializar_proceso, initargs=(CODIGO_SASREC,)) as pool:
        futuros = {pool.submit(mm.correr, base, CATEGORIA_EMB, v, lr, wd, RUTA_NPZ, MAX_EPOCAS): (v, lr, wd)
                   for v, lr, wd in pendientes}
        for n, futuro in enumerate(as_completed(futuros), 1):
            r = futuro.result()
            ruta = archivo(*futuros[futuro])
            with open(ruta + '.tmp', 'w', encoding='utf-8') as f:
                json.dump(r, f, indent=1, ensure_ascii=False, default=float)
            os.replace(ruta + '.tmp', ruta)
            print(f"[{n}/{len(pendientes)}] {r['variante_mm']} lr={r['lr']} wd={r['weight_decay']}: NDCG@10 valid "
                  f"{r['NDCG@10 valid']:.4f} (época {r['época']} de {r['épocas corridas']}) "
                  f"[{(time.time() - inicio) / 60:.1f} min]", flush=True)

## Resultados

Por variante, la combinación con mayor NDCG@10 en `valid` (test no interviene en la elección), con sus métricas de test por id y por título, y separadas por popularidad del ítem objetivo: ≤ 5 interacciones en `train` (usuarios distintos que lo tienen en un historial u objetivo de train) frente al resto. Como referencia, el SASRec BCE (paper) de la corrida local (`baseline_sasrec.ipynb`, celda 25). Se guarda un resumen en `sasrec_mm.json`.

In [ ]:
COLUMNAS = ['lr', 'weight_decay', 'época', 'NDCG@10 valid', 'HR@5 id', 'HR@10 id', 'NDCG@5 id', 'NDCG@10 id',
            'HR@5 título', 'HR@10 título', 'NDCG@5 título', 'NDCG@10 título']
REFERENCIA_LOCAL = {'SASRec BCE (paper), local (sasrec:25)': {
    'HR@5 id': 0.0561, 'HR@10 id': 0.0772, 'NDCG@5 id': 0.0364, 'NDCG@10 id': 0.0432,
    'HR@5 título': 0.0564, 'HR@10 título': 0.0773, 'NDCG@5 título': 0.0368, 'NDCG@10 título': 0.0435,
    'lr': 1e-4, 'weight_decay': 1e-6, 'época': 138}}
combinaciones = []
for nombre_archivo in sorted(os.listdir(CARPETA_MM)):
    if nombre_archivo.endswith('.json'):
        with open(os.path.join(CARPETA_MM, nombre_archivo)) as f:
            combinaciones.append(json.load(f))
todas = pd.DataFrame(combinaciones)
mejores = todas.loc[todas.groupby('variante_mm')['NDCG@10 valid'].idxmax()].set_index('variante_mm')
tabla = pd.concat([mejores[COLUMNAS], pd.DataFrame(REFERENCIA_LOCAL).T])
popularidad = pd.DataFrame({(v, grupo): {'filas': m['filas'], **{k: m[k] for k in m if 'título' in k}}
                            for v, fila in mejores.iterrows() for grupo, m in fila['popularidad'].items()}).T
if not PRUEBA:
    resumen = {'mejores': {v: {k: fila[k] for k in COLUMNAS + ['popularidad', 'épocas corridas', 'segundos']}
                           for v, fila in mejores.iterrows()},
               'combinaciones': len(todas), 'referencia_local': REFERENCIA_LOCAL,
               'embeddings': {k: v for k, v in manifiesto_emb.items() if k != 'entorno'}, 'entorno': entorno()}
    with open(os.path.join(CARPETA_EMB, 'sasrec_mm.json'), 'w', encoding='utf-8') as f:
        json.dump(resumen, f, indent=1, ensure_ascii=False, default=float)
display(tabla.round(4))
popularidad.round(4)

## Exportar (solo Colab)

Arma un zip con la cobertura, los manifiestos y los resultados de SASRec + SigLIP (sin imágenes ni embeddings), lo deja en `MyDrive/IIC3633/export/` y lo descarga. Ese zip es lo que hay que enviar a quien coordina.

In [ ]:
if IN_COLAB:
    from exportar import exportar
    from google.colab import files
    ruta_zip = exportar(RUTAS['iic3633'], 'multimodal')
    files.download(ruta_zip)